# KMeans with 1 variable

## Import Modules

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns

### 🧮 The Math Behind the Elbow Method: Within-Cluster Sum of Squares (WCSS)

To figure out if our clusters are actually meaningful, K-Means calculates the **Total Within-Cluster Sum of Squares (WCSS)**, which is also referred to as **Inertia**. This metric measures how "tight" or compact our clusters are. 

The mathematical formula for WCSS is:

$$WCSS = \sum_{k=1}^{K} \sum_{x_i \in C_k} ||x_i - \mu_k||^2$$

**Let's break down what these symbols mean:**
* $K$ = The total number of clusters.
* $\sum_{k=1}^{K}$ = The outer loop. We are going to add up the results for *every* cluster from $1$ to $K$.
* $\sum_{x_i \in C_k}$ = The inner loop. We are iterating over every individual data point ($x_i$) that belongs to a specific cluster ($C_k$).
* $\mu_k$ = The centroid (the calculated center/mean) of cluster $k$.
* $||x_i - \mu_k||^2$ = The squared Euclidean distance between the data point ($x_i$) and its cluster's center ($\mu_k$).

#### 🧠 What is this equation actually saying?
In plain English: *"For every single cluster, measure the distance from each point to the center of that cluster, square it, and add them all up."* When we build our Elbow Curve, we are plotting this $WCSS$ value on the Y-axis against the number of clusters ($K$) on the X-axis to find the point of diminishing returns!

https://www.naftaliharris.com/blog/visualizing-k-means-clustering/

## Read data

In [ ]:
df = pd.read_csv('data/cluster_1_variable_example.csv')

In [ ]:
df.info()

In [ ]:
df

Let's visualize the observations using a specialized scatter plot The STRIP PLOT.

In [ ]:
sns.relplot(data = df, x="x", y=1, hue="true_group")
plt.show()

In [ ]:
tips = sns.load_dataset("tips")

In [ ]:
tips.info()

In [ ]:
tips.head(10)

In [ ]:
sns.catplot(data=tips, x="total_bill", hue="day", kind="strip", jitter=False, alpha=0.8)

In [ ]:
sns.catplot(data=tips, x="total_bill", hue="day", kind="strip", jitter=True, alpha=0.8)

In [ ]:
sns.relplot(data = df, x='x', y=1, hue='true_group')

plt.show()

In [ ]:
sns.catplot(data = df, x='x', hue='true_group', kind='strip', jitter=False)

plt.show()

Let's pretend we DO NOT know the groupings! Can we CLUSTER the observations just based on their VALUES!

## KMeans

But before we can RUN KMeans...we need to organize the data appopriately. The function we will use WANTS NumPy arrays and NOT Pandas DataFrames.

The DataFrame NEEDS to be a 2D array.

In [ ]:
df.x.to_numpy()

In [ ]:
df.x.to_numpy().reshape(-1, 1)

In [ ]:
X = df.x.to_numpy().reshape(-1, 1)

In [ ]:
X.ndim

In [ ]:
X.shape

In [ ]:
X

The FUNCTION to execute KMeans comes from the scikit-learn MODULE. scikit-learn is RARELY imported directly. Instead, we IMPORT FUNCTIONS from scikit-learn.

The function is named `KMeans()` and it comes from the `sklearn.cluster` module.

In [ ]:
!pip install scikit-learn --break-system-packages

In [ ]:
from sklearn.cluster import KMeans

In [ ]:
%whos

ALL functions from scikit-learn follow the following recipe:

* INITIALIZE the object - specify the ASSUMPTIONS for how the method is executed  
* FIT the object - LEARN what you need to learn  
* PREDICT or TRANSFORM - return the VALUES you want from a FITTED object

Let's begin by INITIALIZING. The ASSUMPTIONS ARE:

* the number of clusters
* the random seed which controls the randomness - the initial guess
* the number of initial guesses
* the number of iterations for each initial guess

Let's begin with 3 clusters since we know there are in fact 3 groups in this small application.

In [ ]:
km3 = KMeans( n_clusters=3, random_state=121, n_init=25, max_iter=500 )

In [ ]:
type( km3 )

We have not provided any DATA when we INITIALIZE!

We give the data when we FIT the object using the assumptions!

In [ ]:
km3 = km3.fit( X )

For KMeans FITTING means it identified the CLUSTERS associated with EACH data point!

To RETURN the CLUSTER ASSIGNMENTS or CLUSTER LABELS...we need to PREDICT the data!!!!

In [ ]:
km3.predict( X )

If you only care about the cluster ASSIGNMENTS or CLUSTER LABELS, FITTING and PREDICTING can be combined into a SINGLE action.

In [ ]:
km3b = KMeans(n_clusters=3, random_state=121, n_init=25, max_iter=500)

In [ ]:
km3b.fit_predict( X )

Accomplish ALL 3 actions in 1 LINE of CODE!!!

In [ ]:
KMeans(n_clusters=3, random_state=121, n_init=25, max_iter=500).fit_predict( X )

Let's use the INITIALIZE then FIT and PREDICT approach and study the cluster labels.

To do that, let's create a COPY of the original dataframe.

In [ ]:
df_copy = df.copy()

In [ ]:
df_copy

Create a new column named `k3` which stores the RESULT of the KMeans cluster labels!

In [ ]:
df_copy['k3'] = pd.Series( km3.fit_predict(X), index=df_copy.index )

In [ ]:
df_copy

In [ ]:
df_copy.info()

In [ ]:
df_copy.nunique()

In [ ]:
df_copy.k3.value_counts()

In [ ]:
df_copy.true_group.value_counts()

Color by the KMeans labels...

In [ ]:
sns.catplot(data = df_copy, x='x', hue='k3', kind='strip', jitter=False)

plt.show()

I like to treat cluster labels as CATEGORICAL variables. Thus, let's CONVERT the integer to the CATEGORY data type!!!!!

The Pandas Category data type is a specialized data type for CATEGORICAL variables!!!!

In [ ]:
df_copy['k3'] = df_copy.k3.astype('category')

In [ ]:
df_copy.info()

In [ ]:
df_copy.k3.cat.categories

In [ ]:
df_copy.k3.value_counts()

In [ ]:
sns.catplot(data = df_copy, x='k3', kind='count')

plt.show()

In [ ]:
sns.catplot(data = df_copy, x='x', hue='k3', kind='strip', jitter=False)

plt.show()

### 



How do the KMeans results compare to the TRUE answers??

You can NEVER do this in a real data set!

However, you are clustering the ROWS based on NUMERIC COLUMNS. Thus, you are NOT using the categorical variables in the clustering! You can therefore compare the CLUSTER results with the known categories!!!

Could use DODGED bar chart, but I prefer to use HEAT MAPS!!!

In [ ]:
pd.crosstab( df_copy.true_group, df_copy.k3 )

But...I like to include the TOTAL or MARGIN COUNTS!!!

In [ ]:
pd.crosstab( df_copy.true_group, df_copy.k3, margins=True )

But lets visualize within a HEAT MAP.

In [ ]:
fig, ax = plt.subplots()

sns.heatmap(data = pd.crosstab( df_copy.true_group, df_copy.k3, margins=True),
            annot=True, annot_kws={'fontsize': 20},
            cbar=False,
            ax=ax )

plt.show()

KMeans has MIS LABELED or MIS IDENTIED 1 of the observations!

But...is KMeans WRONG??? NO!!!!!

KMeans groups by DISTANCE!! The closer observations are the more similar KMeans feels they are!!!

Let's visualize what caused the INCORRECT label...

In [ ]:
sns.catplot(data = df_copy, x='x', hue='k3', kind='strip', jitter=False,
            col='true_group', s=75)

plt.show()

Let's use the marker SHAPE via the `style` argument in `sns.relplot()`.

In [ ]:
sns.relplot(data = df_copy, x='x', y=1, hue='k3', style='true_group', aspect=1.5, s=75)

plt.show()

## More clusters

Let's try 8 clusters!

In [ ]:
km8 = KMeans(n_clusters=8, random_state=121, n_init=25, max_iter=500)

In [ ]:
df_copy['k8'] = pd.Series( km8.fit_predict( X ), index=df_copy.index )

In [ ]:
df_copy.info()

In [ ]:
df_copy.k8.value_counts()

In [ ]:
df_copy['k8'] = df_copy.k8.astype('category')

In [ ]:
df_copy.dtypes

In [ ]:
sns.catplot(data = df_copy, x='x', hue='k8', kind='strip', jitter=False)

plt.show()

The MAX number of clusters equals the NUMBER OF ROWS in the data set!

This would give 1 observation PER cluster!!!!

In [ ]:
km15 = KMeans(n_clusters=15, random_state=121, n_init=25, max_iter=500)

In [ ]:
df_copy['k15'] = pd.Series( km15.fit_predict( X ), index=df_copy.index )

In [ ]:
df_copy.info()

In [ ]:
df_copy.nunique()

In [ ]:
df_copy['k15'] = df_copy.k15.astype('category')

In [ ]:
df_copy.k15.value_counts()

## Optimal number of clusters

We want to find the TOTAL WITHIN SUM of SQUARES that gives "INTERESTING" behavior!!!

We must calculate the TOTAL WITHIN SUM of SQUARES for MANY possible number of clusters!!!!!

The `KMeans()` objects refer to the TOTAL WITHIN SUM of SQUARES as the `.inertia_` attribute.

We want to MINIMIZE the "INERTIA" but only to an "INTERESTING" level!

In [ ]:
tots_within = []

K = range(1, df_copy.shape[0]+1)

for k in K:
    km = KMeans(n_clusters=k, random_state=121, n_init=25, max_iter=500)
    km = km.fit( X )
    
    tots_within.append( km.inertia_ )

In [ ]:
tots_within

Visualize the RESULTS using a LINE CHART.

This is a simple graphic that I typically make in matplotlib rather than Seaborn. The style of the line chart is the SAME style that we used at the beginning semeseter when we plotting the SEM vs the sample size.

In [ ]:
fig, ax = plt.subplots()

ax.plot( K, tots_within, 'bo-')
ax.set_xlabel('number of clusters')
ax.set_ylabel('total within sum of squares')

plt.show()

# KMeans with 2 variables

## Import Modules

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns

## Read data

In [ ]:
df = pd.read_csv('data/cluster_2_variables_example.csv')

In [ ]:
df.info()

In [ ]:
df.dtypes

In [ ]:
df.isna().sum()

In [ ]:
df.nunique()

In [ ]:
df.true_group.value_counts()

In [ ]:
sns.relplot(data = df, x='x1', y='x2')

plt.show()

Color by the TRUE KNOWN groupings.

In [ ]:
sns.relplot(data = df, x='x1', y='x2', hue='true_group')

plt.show()

## KMeans

Let's prepare the data as required by scikit-learn.

In [ ]:
X = df.select_dtypes('number').copy().to_numpy()

In [ ]:
X.shape

In [ ]:
from sklearn.cluster import KMeans

INITIIALIZE, FIT, and PREDICT!!!

But let's do ALL ACTIONS in a single line of code!!

We know there are 5 groupings, so let's use 5 clusters!

In [ ]:
clusters_5 = KMeans( n_clusters=5, random_state=121, n_init=25, max_iter=500 ).fit_predict( X )

Now, assign the CLUSTER LABELS to a copied DataFrame.

In [ ]:
df_copy = df.copy()

In [ ]:
df_copy['k5'] = pd.Series( clusters_5, index=df_copy.index ).astype('category')

In [ ]:
df_copy.info()

In [ ]:
df_copy.nunique()

In [ ]:
df_copy.k5.value_counts()

In [ ]:
df_copy.true_group.value_counts()

Let's visualize the cluster results on a scatter plot.

In [ ]:
sns.relplot(data = df_copy, x='x1', y='x2', hue='k5')

plt.show()

Let's use a HEAT MAP to COMPARE the cluster labels with the KNOWN true groupings!

In [ ]:
fig, ax = plt.subplots()

sns.heatmap( data = pd.crosstab( df_copy.true_group, df_copy.k5, margins=True ),
             annot=True, annot_kws={'fontsize': 15},
             cbar=False,
             ax=ax )

plt.show()

Let's use the Marker SHAPE via the `style` argument in `sns.relplot()` to check which specific data points are mis labeled.

In [ ]:
sns.relplot(data = df_copy, x='x1', y='x2', hue='k5', style='true_group')

plt.show()

KMeans has NO KNOWLEDGE of the groupings! It's clustering ONLY based on the numeric columns!!!!!!!!!!

## What if we did NOT know the number of clusters to try?

I always recommend starting out with 2 clusters!!! You should do this just to see what happens!!!!

In [ ]:
clusters_2 = KMeans(n_clusters=2, random_state=121, n_init=25, max_iter=500).fit_predict( X )

In [ ]:
df_copy['k2'] = pd.Series( clusters_2, index=df_copy.index ).astype('category')

In [ ]:
df_copy.info()

In [ ]:
df_copy.k2.value_counts()

In [ ]:
sns.relplot(data = df_copy, x='x1', y='x2', hue='k2')

plt.show()

## Optimal number of clusters

You do NOT need to manually try out many different number of clusters.

Try 2 clusters. Then try one more representative number based on your exploration OR based on some known grouping.

Then use the KNEE BEND plot to figure out a "better" number of clusters.

In [ ]:
tots_within = []

K = range(1, 31)

for k in K:
    km = KMeans(n_clusters=k, random_state=121, n_init=25, max_iter=500)
    km = km.fit( X )
    
    tots_within.append( km.inertia_ )

Visualize the KNEE BEND plot as a line chart.

In [ ]:
fig, ax = plt.subplots()

ax.plot( K, tots_within, 'bo-' )
ax.set_xlabel('number of clusters')
ax.set_ylabel('total within sum of squares')

plt.show()